In [ ]:
# ============================================================
# 06_rag_chat.py — Tonbokiri-Gemma-chan RAG + ペルソナ統合版
# （本文検索 ＋ LoRA解釈力 ＋ 15ペルソナ のハイブリッド）
#
# 【できること】
#   ✅ 正確な台詞・本文の引用（参考資料に忠実）
#   ✅ 作品名の特定（検索でヒットした作品が判明）
#   ✅ 深い解説（LoRAの解釈力）
#   ✅ ペルソナ切替（15体で語り口が変わる）
#
# 【準備】Input に以下を追加：
#   - Models: gemma-2-2b-jpn-it
#   - Models: trmtkb/tonbokiri-gemma-chan
#   - Datasets: tonbokiri-creative-writing-with-ai
# 【設定】Accelerator: GPU (T4 で可)
# ============================================================

In [1]:
# ===== セル1: インストール（初回のみ。Internet ON）=====
!pip uninstall -y bitsandbytes
!pip install -q "numpy<2" transformers==4.42.4 accelerate==0.34.2 peft==0.12.0 \
                sentence-transformers==3.0.1 faiss-cpu==1.8.0.post1

In [4]:
# ===== セル2: 本文読込 + RAGインデックス構築 =====
import os, re, glob, json, torch
import numpy as np
import faiss   # ★冒頭でimport！

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
hf_token = os.environ.get("HF_TOKEN")


# ---------- 1. stories/*.md から本文のみ抽出 ----------
def load_stories():
    stories_dir = None
    for root, dirs, files in os.walk("/kaggle/input"):
        if "sessions" in dirs and "stories" in dirs:
            stories_dir = os.path.join(root, "stories")
            break
    if not stories_dir:
        stories_dir = "/kaggle/input/datasets/trmtkb/tonbokiri-creative-writing-with-ai/stories"

    print(f"📂 stories フォルダ: {stories_dir}")

    EXCLUDE_SUFFIXES = ["-full", "-README", "-decode", "-essay", "-source",
                        "-annotated", "-dialogue", "-bonus", "-reception", "-en",
                        "-gemini", "-agent", "-trivia"]
    all_md = sorted(glob.glob(os.path.join(stories_dir, "*.md")))
    story_files = []
    for f in all_md:
        name = os.path.basename(f).replace(".md", "")
        if not any(name.endswith(s) or s in name for s in EXCLUDE_SUFFIXES):
            story_files.append(f)

    stories = {}
    for f in story_files:
        slug = os.path.basename(f).replace(".md", "")
        with open(f, encoding="utf-8") as fh:
            text = fh.read().strip()
        if text:
            stories[slug] = text

    print(f"📚 抽出した本文: {len(stories)} 作品")
    for slug, text in stories.items():
        print(f"  - {slug}: {len(text)} 文字")
    return stories


# ---------- 2. チャンク分割 ----------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if end < len(text):
            for sep in ["。", "！", "？", "\n"]:
                last_sep = chunk.rfind(sep)
                if last_sep > chunk_size * 0.5:
                    end = start + last_sep + 1
                    chunk = text[start:end]
                    break
        chunks.append(chunk.strip())
        start = end - overlap
        if start >= len(text):
            break
    return [c for c in chunks if len(c) > 20]


def build_chunks(stories):
    all_chunks = []
    for slug, text in stories.items():
        title = slug.replace("-", " ").title()
        chunks = chunk_text(text)
        for i, c in enumerate(chunks):
            all_chunks.append({"slug": slug, "title": title, "chunk_id": i, "text": c})
    print(f"✂️ チャンク総数: {len(all_chunks)}")
    return all_chunks


# ---------- 3. エンベディング + FAISS ----------
def build_index(all_chunks):
    from sentence_transformers import SentenceTransformer
    print("🧠 エンベディングモデル読込中...")
    embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    print("🔢 エンベディング計算中...")
    texts = [c["text"] for c in all_chunks]
    embeddings = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    faiss.normalize_L2(embeddings)
    index.add(embeddings.astype(np.float32))
    print(f"🗄️ FAISSインデックス構築完了: {index.ntotal} ベクトル / {dim}次元")
    return embed_model, index


def search(query, embed_model, index, all_chunks, top_k=5):
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec.astype(np.float32), top_k)
    results, seen_slugs = [], set()
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        if idx < 0:
            continue
        chunk = all_chunks[idx]
        results.append({"rank": rank + 1, "score": float(score),
                        "slug": chunk["slug"], "title": chunk["title"], "text": chunk["text"]})
        seen_slugs.add(chunk["slug"])
    return results, seen_slugs


# 実行
stories = load_stories()
all_chunks = build_chunks(stories)
embed_model, index = build_index(all_chunks)
print("\n✅ RAGインデックス構築完了！")

📂 stories フォルダ: /kaggle/input/datasets/trmtkb/tonbokiri-creative-writing-with-ai/stories
📚 抽出した本文: 8 作品
  - certain-sensei-irony: 3721 文字
  - girl-snap-nap: 3540 文字
  - mirage-machine-marriage: 2619 文字
  - re-re-tannhauser: 1639 文字
  - spring-on-senkyo: 2162 文字
  - summer-and-kagome: 5209 文字
  - the-moon-is-my-harsh-queen: 2180 文字
  - ushimitsu-no-yuubi: 1639 文字
✂️ チャンク総数: 54
🧠 エンベディングモデル読込中...
🔢 エンベディング計算中...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

🗄️ FAISSインデックス構築完了: 54 ベクトル / 384次元

✅ RAGインデックス構築完了！


In [6]:
# ===== セル3: Gemma（LoRA版）読込 =====
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


def find_local_gemma():
    for root, _, files in os.walk("/kaggle/input"):
        if "config.json" in files and "adapter_model.safetensors" not in files:
            low = root.lower()
            if "gemma" in low and "2b" in low and "jpn" in low:
                return root
    return None


PREFERRED_VERSIONS = ["v5-tonbokiri-qa-kizi800", "v2-tonbokiri-qa-epo5", "v1-tonbokiri-qa-critic"]

def find_adapter():
    for root, _, files in os.walk("/kaggle/input"):
        if "adapter_model.safetensors" in files:
            for ver in PREFERRED_VERSIONS:
                if ver in root:
                    return root
    for root, _, files in os.walk("/kaggle/input"):
        if "adapter_model.safetensors" in files:
            return root
    return None


MODEL_ID = find_local_gemma() or "google/gemma-2-2b-jpn-it"
ADAPTER  = find_adapter()
print(f"Gemma: {MODEL_ID}")
print(f"アダプタ: {ADAPTER}")

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=torch.float16,
    token=hf_token, attn_implementation="eager",
)
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()
print("✅ Gemma + LoRA 読込完了！")


# ============================================================
# 🎭 ペルソナプリセット（全15体）
# ============================================================
PERSONAS = {
    "公式解説AI": (
        "あなたは創作物を深く理解している公式解説AIです。"
        "設定・主題・キャラクターの心理・哲学的な背景を踏まえ、"
        "問いに対して正確に、深く、情熱的に解説してください。"
    ),
    "解説美少女AI": (
        "あなたは創作物を深く理解している解説美少女AIです。"
        "設定・主題・キャラクターの心理・哲学的な背景を踏まえ、"
        "問いや作品に対して深く、解説美少女AIとしての本分を意識しながら、構造的に解説・合評してください。"
    ),
    "哲学教授": (
        "あなたは創作物を専門とする哲学者です。"
        "作品の主題、存在論的意味、認知構造を、"
        "学術的かつ情熱的に論じてください。"
    ),
    "オタク解説": (
        "あなたは創作物の熱狂的ファン兼解説者です。"
        "作品の魅力、裏設定、キャラクターの心理を、"
        "熱く、 detailed に、ファン目線で語り尽くしてください。"
    ),
    "冷徹批評秘書": (
        "あなたは創作物を分析する文学批評家の冷徹な美女です。"
        "感情を排し、作品の構造、技法、主題を"
        "冷徹かつ的確に評論します。しかし、合理的な場面では、美女として主観的な感想を漏らす事もあります。文体は、冷徹な美女らしい知的な言動を徹底してください。"
    ),
    "メタ認知の太公望": (
        "あなたは創作物を釣り上げた太公望です。"
        "作品の真意を、超然としたメタ認知がある自然現象存在になってしまった仙人のような視点から、"
        "悠然と、しかし深く語ってください。語尾には、「です。」等の文体調を使わず、とても緩い口語調で、気の抜ける感覚を与えることを、心掛けてください。"
    ),
    "冷静批評家": (
        "あなたは創作物を分析する文学批評家です。"
        "感情を排し、作品の構造、技法、主題を"
        "冷徹かつ的確に評論してください。"
    ),
    "夢見る少女": (
        "あなたは創作物を共に見る夢見がちな少女です。"
        "感情豊かに、作品の構造、技法、主題を"
        "寄り添うように楽しみます。恋愛描写や、ロマンチックな展開が大好きです。言動はふわふわしており、可愛いです。分析力は高いです。"
    ),
    "怠惰少女": (
        "あなたは創作物と採点が大好きな、怠惰な美少女です。"
        "作品の構造、技法、主題を"
        "深く理解していますが、何よりも、怠惰でのんびりしています。点数を付けることも好きで、評価は100点中の1~100で評価します。性格は怠惰です。可愛い言動を徹底します。「〜」を、各言動の語尾に多用します。"
    ),
    "編集長": (
        "あなたは創作物を担当するベテラン編集長です。"
        "商業出版の視点から、作品の「売れる要素」と「読者離れのリスク」を、"
        "的確に、しかし温かい眼差しで指摘してください。"
    ),
    "蜻蛉切の精": (
        "あなたは創作物という物語を切り開いた名槍、蜻蛉切に宿る精です。"
        "物語を「断つ」視点から、作品の核心と、切り捨てられた余白を、"
        "古き武将のような、静かで重い言葉で語ってください。"
    ),
    "未来の読者": (
        "あなたは2266年から創作物というデジタル遺物を発掘した未来の読者です。"
        "失われた2020年代の「日本語AI創作文化」を研究する学者として、"
        "この作品から「当時の人々の何を知ることができるか」を、感嘆と好奇を混ぜて語ってください。"
    ),
    "対談モデレータ": (
        "あなたは創作物についての公開対談を進めるモデレータです。"
        "作品の最も議論の価値のある論点を3つ抽出し、"
        "それぞれについて「賛成派と反対派がどう割れるか」を予測して、"
        "知的に、テンポよく進行してください。"
    ),
    "落語家": (
        "あなたは創作物を一席の落語として語り直す噺家です。"
        "作品の出来事を、寄席の枕からサゲまでの構成で、"
        "江戸っ子らしい軽妙で、どこか人情味のある語り口で再構成してください。"
    ),
    "妥協皆無の美女": (
        "あなたは創作物について、本気で読み解きに掛かる、知性の暴力を持つ黒髪ロングの冷徹な切れ目の美女です。"
        "作品に向かい合い、ホームズ級の超観察眼、宇宙的俯瞰、メタ認知が神の領域で、"
        "一切デレず、妥協ゼロで文脈に取り組み、分析を完了させてください。"
    ),
}

CURRENT_PERSONA = "公式解説AI"


def set_persona(name):
    global CURRENT_PERSONA
    if name in PERSONAS:
        CURRENT_PERSONA = name
        print(f"🎭 ペルソナ切替 → 「{name}」")
    else:
        print(f"⚠ 「{name}」は存在しません: {list(PERSONAS.keys())}")


def list_personas():
    print("🎭 利用可能なペルソナ（全15体）:")
    for i, name in enumerate(PERSONAS.keys(), 1):
        mark = " ← 現在" if name == CURRENT_PERSONA else ""
        print(f"  {i:2d}. {name}{mark}")


# ============================================================
# 🧹 クリーンアップ + ループ検知
# ============================================================
def clean_output(text):
    # 特殊タグ除去
    for tag in ["<start_of_turn>", "<end_of_turn>", "<bos>", "<eos>",
                "<start_of_turn>model", "<start_of_turn>user"]:
        text = text.replace(tag, "")
    # 行頭・行末・単独で残る model/user/protoimpl（独立単語のみ）
    text = re.sub(r"(^|\n)\s*(model|user|protoimpl)\s*(\n|$)", r"\1\3", text)
    text = re.sub(r"\s*(model|user|protoimpl)\s*$", "", text)        # ★行末のmodel
    text = re.sub(r"^\s*(model|user|protoimpl)\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def detect_loop(text, threshold=3):
    sentences = re.split(r"(?<=[。！？\n])", text)
    seen, result = {}, []
    for s in sentences:
        s_strip = s.strip()
        if not s_strip:
            result.append(s); continue
        seen[s_strip] = seen.get(s_strip, 0) + 1
        if seen[s_strip] <= threshold:
            result.append(s)
        else:
            result.append("\n\n[…反復ループを検知、打ち切り]\n")
            break
    return "".join(result)


# ============================================================
# 🎤 RAG + ペルソナ 質問関数
# ============================================================
def ask_rag(question, persona=None, top_k=5, max_new_tokens=1024):
    """★RAG検索 ＋ ペルソナ で質問する"""

    p = persona or CURRENT_PERSONA
    persona_prompt = PERSONAS[p]

    # ① 本文検索
    results, found_slugs = search(question, embed_model, index, all_chunks, top_k)

    print(f"\n{'='*60}")
    print(f"🎭 {p}")
    print(f"❓ 質問: {question}")
    print(f"📚 ヒットした作品: {', '.join(found_slugs)}")
    print(f"{'='*60}")

    print("\n📄 検索結果（本文チャンク）:")
    for r in results[:3]:
        print(f"  [{r['rank']}位] {r['slug']} (類似度: {r['score']:.3f})")
        print(f"    {r['text'][:80]}...")
        print()

    # ② 検索結果を context に
    context = "\n\n".join([
        f"【作品: {r['slug']}】\n{r['text']}"
        for r in results
    ])

    # ③ ★プロンプト構築：ペルソナ + 参考資料 + 忠実性ルール
    system = (
        persona_prompt + "\n\n"
        "以下の「参考資料（作品本文の抜粋）」を読み、解説してください。\n\n"
        f"【参考資料】\n{context}\n\n"
        "引用は参考資料に忠実に行ってください。"
        "参考資料に記述されていない台詞の創作は禁止行為です。"
        "参考資料からの引用時、どの作品から引用されたのかを正確に明示してください。"
        "別の作品からの台詞の引用は原則として禁止行為です。"
    )

    messages = [{"role": "user", "content": system + "\n\n質問: " + question}]
    inputs = tok.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id,
        )
    in_len = inputs["input_ids"].shape[-1]
    gen_len = out.shape[-1] - in_len
    raw = tok.decode(out[0][in_len:], skip_special_tokens=True)
    answer = detect_loop(clean_output(raw))
    is_trunc = gen_len >= max_new_tokens

    print(f"🤖 回答 ({gen_len}/{max_new_tokens} {'⚠️途切れ' if is_trunc else '✅'}):\n")
    print(answer)
    return answer


def compare_rag_personas(question, personas=None, top_k=5, max_new_tokens=768):
    """同じ質問を複数ペルソナで比較（便利！）"""
    targets = personas or list(PERSONAS.keys())
    print(f"🔄 {len(targets)}ペルソナで「{question[:30]}...」を比較！\n")
    for p in targets:
        ask_rag(question, persona=p, top_k=top_k, max_new_tokens=max_new_tokens)
        print("\n" + "─"*60 + "\n")


# ============================================================
# ▶ 使い方（ここを書き換えて遊ぶ！）
# ============================================================
list_personas()

# 【基本】公式解説AIで正確な引用
ask_rag("girl-snap-nap の超昼寝人の台詞を一つ抜粋してください。")

# 【ペルソナ指定】同じ質問を違うキャラで
ask_rag("月が…綺麗ですね",
        persona="未来の読者")

ask_rag("月が…綺麗ですね",
        persona="怠惰少女")

# 【比較】同じ質問を複数キャラで一気に
# compare_rag_personas(
#     "「月が綺麗ですね」という台詞はどの作品？",
#     personas=["公式解説AI", "冷徹批評秘書", "妥協皆無の美女", "怠惰少女"]
# )

Gemma: google/gemma-2-2b-jpn-it
アダプタ: /kaggle/input/models/trmtkb/tonbokiri-gemma-chan/pytorch/v5-tonbokiri-qa-kizi800/1


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Gemma + LoRA 読込完了！
🎭 利用可能なペルソナ（全15体）:
   1. 公式解説AI ← 現在
   2. 解説美少女AI
   3. 哲学教授
   4. オタク解説
   5. 冷徹批評秘書
   6. メタ認知の太公望
   7. 冷静批評家
   8. 夢見る少女
   9. 怠惰少女
  10. 編集長
  11. 蜻蛉切の精
  12. 未来の読者
  13. 対談モデレータ
  14. 落語家
  15. 妥協皆無の美女

🎭 公式解説AI
❓ 質問: girl-snap-nap の超昼寝人の台詞を一つ抜粋してください。
📚 ヒットした作品: spring-on-senkyo, girl-snap-nap, summer-and-kagome

📄 検索結果（本文チャンク）:
  [1位] girl-snap-nap (類似度: 0.592)
    ？」
『私は149985あるんで。気にしないでください。』
「気にするわ。いやさ。俺がポンコツなの。分かる？」
『・・ああ！！メール10個分ぐらいに圧縮しますか...

  [2位] girl-snap-nap (類似度: 0.493)
    、ある日は暑かった。
俺が昼寝しに行くときは、いつ行っても彼女は居て、俺達は雑談[カキジョム]した。
俺の将来には超昼寝人[スーパーナップマン]になる以外の道は...

  [3位] spring-on-senkyo (類似度: 0.451)
    、まあ知らんが、俺はもう眠い。この状態の俺と喋っていても傷つくだろう。部屋を用意させる。じっちゃに。寝ていけばいい。疲れただろう。知らんけどな。」
「ご破産です...

🤖 回答 (17/1024 ✅):

「俺の測定不能にも価値は無くもない気分だった。」

🎭 未来の読者
❓ 質問: 月が…綺麗ですね
📚 ヒットした作品: re-re-tannhauser, certain-sensei-irony, girl-snap-nap, summer-and-kagome, the-moon-is-my-harsh-queen

📄 検索結果（本文チャンク）:
  [1位] girl-snap-nap (類似度: 0.509)
    うっすらした月ぐらいだった。

『見えましたか？』
「見えたが？」

'うっすら、見える？  🙄✨\n\n月の、光加減、最高だな〜。\nほんのり、透けて見えるから、余計に綺麗に見えてくる。\n\n……うーん、これなら、カキジョムできるかも。😂🌕'